# Owen Coder DPO Alignment Training

Direct Preference Optimization (DPO) to align Owen Coder's responses:
- Prefer detailed vulnerability analysis over vague warnings
- Prefer actionable fixes over generic advice
- Prefer correct CWE mappings over approximate ones
- Prefer acknowledging clean code over false positives

**Prerequisites:** Upload `training_data_merged.jsonl` to Colab.

**Base model:** Use the SFT model from Owen_Coder_Expert.ipynb or Owen_Coder_7B.ipynb

In [ ]:
# Install dependencies
!pip install -q unsloth trl datasets transformers peft accelerate bitsandbytes
!pip install -q sentencepiece protobuf

In [ ]:
import json
import random
import torch
from datasets import Dataset
from unsloth import FastLanguageModel

# Config
BASE_MODEL = "unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit"  # Or 7B
SFT_ADAPTER = "/content/drive/MyDrive/owen_coder/sft_adapter"  # Path to SFT adapter
MAX_SEQ_LENGTH = 2048
LORA_R = 16
LORA_ALPHA = 16

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Load model with SFT adapter
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

# Apply LoRA for DPO
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

print(f"Model loaded. Trainable params: {model.print_trainable_parameters()}")

In [ ]:
# Build DPO preference pairs from training data
# Each pair: (prompt, chosen_response, rejected_response)

def build_dpo_pairs(data_path="training_data_merged.jsonl"):
    """Create preference pairs for DPO training."""
    with open(data_path) as f:
        data = [json.loads(l) for l in f if l.strip()]
    
    pairs = []
    random.seed(42)
    
    # Type 1: Detailed analysis (chosen) vs vague response (rejected)
    for item in data:
        output = item["output"]
        if "CWE-" not in output and "SAFE" not in output.upper():
            continue
        
        # Build rejected: strip CWE, strip code blocks, be vague
        vague = output.split("\n")[0]  # Just the first line
        if len(vague) < 20:
            vague = "This code may have security issues. Consider reviewing it."
        
        pairs.append({
            "prompt": item["instruction"],
            "chosen": output,
            "rejected": vague,
        })
    
    # Type 2: Correct CWE (chosen) vs wrong CWE (rejected)
    cwe_swaps = {
        "CWE-89": "CWE-79", "CWE-79": "CWE-89",
        "CWE-78": "CWE-22", "CWE-22": "CWE-78",
        "CWE-502": "CWE-89", "CWE-918": "CWE-79",
    }
    for item in data:
        output = item["output"]
        for correct, wrong in cwe_swaps.items():
            if correct in output:
                wrong_output = output.replace(correct, wrong)
                pairs.append({
                    "prompt": item["instruction"],
                    "chosen": output,
                    "rejected": wrong_output,
                })
                break
    
    # Type 3: Clean code correctly identified (chosen) vs false positive (rejected)
    safe_items = [i for i in data if "no vulnerabilit" in i["output"].lower() 
                  or "secure" in i["output"].lower() or "safe" in i["output"].lower()]
    for item in safe_items:
        false_positive = "**SQL Injection** (CWE-89) [HIGH]\n\nThis code appears to use string concatenation in queries."
        pairs.append({
            "prompt": item["instruction"],
            "chosen": item["output"],
            "rejected": false_positive,
        })
    
    random.shuffle(pairs)
    return pairs

dpo_pairs = build_dpo_pairs()
print(f"Built {len(dpo_pairs)} DPO preference pairs")

In [ ]:
# Format as chat messages for DPO
def format_chat(prompt, response):
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt},
         {"role": "assistant", "content": response}],
        tokenize=False
    )

dpo_dataset = Dataset.from_dict({
    "prompt": [format_chat(p["prompt"], "").rsplit("<|im_start|>assistant", 1)[0] + "<|im_start|>assistant\n" for p in dpo_pairs],
    "chosen": [p["chosen"] for p in dpo_pairs],
    "rejected": [p["rejected"] for p in dpo_pairs],
})

print(f"DPO dataset: {len(dpo_dataset)} examples")
print(f"Example prompt: {dpo_dataset[0]['prompt'][:200]}...")

In [ ]:
# DPO Training
from trl import DPOTrainer, DPOConfig

dpo_config = DPOConfig(
    output_dir="./dpo_output",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    beta=0.1,  # DPO beta — lower = stronger preference
    max_length=MAX_SEQ_LENGTH,
    max_prompt_length=MAX_SEQ_LENGTH // 2,
    logging_steps=10,
    save_steps=100,
    warmup_steps=20,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    optim="adamw_8bit",
    report_to="none",
)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,  # Implicit reference model
    args=dpo_config,
    train_dataset=dpo_dataset,
    tokenizer=tokenizer,
)

print("Starting DPO training...")
dpo_trainer.train()
print("DPO training complete!")

In [ ]:
# Test the DPO model
FastLanguageModel.for_inference(model)

test_prompts = [
    'Review this Python code:\n```python\ndb.execute(f"SELECT * FROM users WHERE name=\'{name}\'")',
    'Is this code safe?\n```python\ndb.execute("SELECT * FROM users WHERE name=?", (name,))\n```',
    'What CWE does this have?\n```javascript\nelement.innerHTML = userInput;\n```',
]

for prompt in test_prompts:
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    outputs = model.generate(input_ids=inputs, max_new_tokens=512, temperature=0.1)
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    print(f"Q: {prompt[:80]}...")
    print(f"A: {response[:300]}")
    print("---")

In [ ]:
# Save adapter and export GGUF
model.save_pretrained("owen_coder_dpo_adapter")
tokenizer.save_pretrained("owen_coder_dpo_adapter")

# GGUF export
model.save_pretrained_gguf(
    "owen_coder_dpo_gguf",
    tokenizer,
    quantization_method="q4_k_m"
)
print("GGUF exported!")

In [ ]:
# Copy to Google Drive
from google.colab import drive
drive.mount("/content/drive")

import shutil, os
dest = "/content/drive/MyDrive/owen_coder/dpo"
os.makedirs(dest, exist_ok=True)

# Copy adapter
shutil.copytree("owen_coder_dpo_adapter", f"{dest}/adapter", dirs_exist_ok=True)

# Copy GGUF
import glob
for f in glob.glob("owen_coder_dpo_gguf/*.gguf"):
    shutil.copy2(f, dest)
    print(f"Copied {f} -> {dest}")

print(f"\nAll DPO artifacts saved to {dest}")
!ls -la {dest}